<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/OnBeautyContests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beauty Contests

A beauty contest game is one which contains an element of "trying to be like the others," which usually means that each player is trying to take an action that is close to the actions of others. A simple setup is as follows:

Player $i$'s utility loss is:

$$
L_i = a(x_i-t_i)^2 + b(x_i-\bar x)^2
$$

Where $x_i \in (-\infty,\infty)$ is the player's action, $a$ and $b$ are positive parameters, and $t_i$ is some given target level of action. $\bar x$ is the average action taken by all players. My notation isn't great, but since this is a loss function, players want to minimize it.

An interpretation of this is that if others were not a concern, players would just choose $x_i=t_i$. Alternatively, if $a=0$, players would try to choose the average action of everyone else.

These sorts of games in part form the basis of _peer effect_ style games. A great example of these sorts of games is in the classic paper **Linear Social Interactions Models**, by Blume, Brock, Durlauf, and Jayaraman.

Let's solve such a model with two players and see what issues it presents.

In [1]:
from sympy import *

x1, x2, t1, t2, a, b, = symbols('x1 x2 t1 t2 a b')
l1 = a*(x1-t1)**2 + b*(x1-x2)**2
l2 = a*(x2-t2)**2 + b*(x2-x1)**2

In the above code, we just introduce actions, targets, and utility penalties from taking an action different from the other player. Note that these "utilities" are really costs, and we are trying to minimize the expressions $u_1$ and $u_2$. We find the Nash equilibrium by the usual method:

In [3]:
f1 = diff(l1, x1)
f2 = diff(l2, x2)
nesolns = solve([f1, f2], [x1, x2])
nesolns[x1].simplify()

(a*t1 + b*t1 + b*t2)/(a + 2*b)

THe solution for player two's optimal action is similar. A point of interest is that the private solutions at the nash equilibrium might diverge from the socially optimal decisions, which maximize the sum of utilities - these are values for $x_1$ and $x_2$ that players would agree on if they had some communication and enforcement mechanism. [Side question: would players want to cheat on such an agreement?]

In [4]:
g1 = diff(l1+l2, x1)
g2 = diff(l1+l2, x2)
sosolns = solve([g1, g2], [x1, x2])
sosolns[x1].simplify()

(a*t1 + 2*b*t1 + 2*b*t2)/(a + 4*b)

To see how these are different, let's just plug in some numbers, like $a=1, b=1, t_1=0, t_2=1$.

In [7]:
parts = {a:1,b:1,t1:0,t2:1}
nesolns[x1].subs(parts), nesolns[x2].subs(parts)

(1/3, 2/3)

In [8]:
sosolns[x1].subs(parts), sosolns[x2].subs(parts)

(2/5, 3/5)

We see that the socially optimal solutions are closer to the center and farther from each players' preferred outcome.

## Morris and Shin

Morris and Shin have another take on a beauty contest game, and I'll put it together as follows. Suppose some random variable of interest - call its realized value $\theta$ -  is drawn from a distribution which everyone knows. Suppose the prior distribution is mean $z$ with standard deviation $\tau$. Agents want to be near the actual value of this random variable, but also want to be near the average. They develop a nice welfare function, but I'm going to leave that out of this just so you get a feel for the solution method.

Each agent gets a private signal of the true value of $\theta$, which is normal around $\theta$ with standard deviation $\sigma$.

Morris and Shin develop a model that has each agent trying to set their actions to the value of $\theta$ (i.e., to match the state), but agents also want to set their action close to the actions of everyone else. Maybe you are going to a dinner party, and have received some idea as to what you should wear (i.e., a tuxedo) but you also don't want to show up at the party wearing a tuxedo if noone else is!

In the end, this results in my optimal action being a weighted average of the state, and the average action of everyone else; actually, my beliefs about these two things based on my private signal $x_i$.

$$
a_i = rE[\theta|x_i]+(1-r)E[\bar a|x_i]
$$

The first part is fairly easy - using standard Bayesian updating my beliefs about the parameter $\theta$ given my signal is that it has expectation:

$$
E[\theta|x_i] = \frac{\sigma^2 z+\tau^2 x_i}{\sigma^2 + \tau^2}
$$

What do I think about the average actions of others? Well, suppose that I think they follow a _linear strategy_:

$$
a_j = \kappa x_j + (1-\kappa) z
$$

Now, on average, I think that these individuals receive a signal equal to the value of $\theta$, so we might say my guess is that the average action obeys the above rule, where I plug in my beliefs about the average signal everyone else gets:

$$
E[\bar a] = \kappa E[\theta|x_i] + (1-\kappa)z
$$
Plugging all of this into my optimal action function, we have:

$$
a_i = rE[\theta|x_i]+(1-r)\left(\kappa E[\theta|x_i]+(1-\kappa) z\right)
$$

So, we find that:

$$
a_i = (r + (1-r)\kappa) E[\theta|x_i]+(1-\kappa)z
$$

Or

$$
a_i = (r + (1-r)\kappa)\frac{\sigma^2 z+\tau^2 x_i}{\sigma^2 + \tau^2}+(1-\kappa)z
$$

But in equilibrium, it must be the case that $i$'s strategy is consistent with the strategy posited for everyone.  Hence, we must have:

$$
\kappa = (r+(1-r)\kappa)\frac{\tau^2}{\tau^2+\sigma^2}
$$


In [10]:
kappa, r, tau, sigma = symbols('kappa r tau sigma')

eq = kappa - (r+(1-r)*kappa)*tau**2/(tau**2 + sigma**2)
kapeq = solve(eq, kappa)[0]
kapeq.simplify()

r*tau**2/(r*tau**2 + sigma**2)

So, we have solved for the equilibrium effort. MOrris and Shin show that agents overemphasize public information, but you should go read about that.
